![head.png](https://github.com/cafawo/FinancialDataAnalytics/blob/master/figures/head.jpg?raw=1)

# Financial Data Analytics in Python

**Prof. Dr. Fabian Woebbeking**</br>
Assistant Professor of Financial Economics

IWH - Leibniz Institute for Economic Research</br>
MLU - Martin Luther University Halle-Wittenberg

fabian.woebbeking@iwh-halle.de

# Homework: data management

You will need a Git/GitHub repository to submit your course deliverables. Consult [**slides.ipynb**](https://github.com/cafawo/FinancialDataAnalytics) for help with the tasks below! If you need further assistance, do not hesitate to open a Q&A at https://github.com/cafawo/FinancialDataAnalytics/discussions

### Task: 

Use Deribit's `"/public/get_tradingview_chart_data"` API endpoint to download historical price data for **"BTC-PERPETUAL"** with a resolution of $60$ minutes, reaching as far back as possible.

Hint: https://docs.deribit.com/#public-get_tradingview_chart_data

In [ ]:
import requests
import pandas as pd
import time

url = "https://www.deribit.com/api/v2/public/get_tradingview_chart_data"

params = {
    "instrument_name": "BTC-PERPETUAL",
    "start_timestamp": 1451606400000,  # 2016-01-01
    "end_timestamp":   int(time.time() * 1000),  # now
    "resolution":      "60",
}

response = requests.get(url, params=params)
result = response.json()["result"]

df = pd.DataFrame({
    "datetime": pd.to_datetime(result["ticks"], unit="ms", utc=True),
    "open":     result["open"],
    "high":     result["high"],
    "low":      result["low"],
    "close":    result["close"],
    "volume":   result["volume"],
})

print(df)
df.to_csv("BTC_PERPETUAL_60m.csv", index=False)

df.to_csv("C:/Users/meder/Documents/Uni Halle Studieninhalte/Python/FinancialDataAnalytics/homework/homework_submitted/08-django_BTC_PERPETUAL_60m.csv", index=False)

                      datetime     open     high      low    close      volume
0    2025-12-02 03:00:00+00:00  86554.5  87174.5  86221.0  86999.0  209.233762
1    2025-12-02 04:00:00+00:00  86999.0  87280.0  86920.5  86987.0  206.927476
2    2025-12-02 05:00:00+00:00  86986.5  87138.0  86849.0  87012.5  117.700418
3    2025-12-02 06:00:00+00:00  87027.5  87201.5  86874.0  87121.5  172.821149
4    2025-12-02 07:00:00+00:00  87116.5  87134.0  86901.5  87020.5  242.420860
...                        ...      ...      ...      ...      ...         ...
4996 2026-06-28 07:00:00+00:00  59943.0  60199.5  59939.0  60149.5  189.228373
4997 2026-06-28 08:00:00+00:00  60149.0  60472.0  60080.0  60305.0  150.632068
4998 2026-06-28 09:00:00+00:00  60302.0  60412.5  60195.0  60195.0   68.155491
4999 2026-06-28 10:00:00+00:00  60194.0  60230.0  60010.0  60092.0   81.388540
5000 2026-06-28 11:00:00+00:00  60092.5  60295.0  60048.0  60227.5   48.590531

[5001 rows x 6 columns]


### Task: 

Create a Pandas data frame called "ohlc" with the price data from above. Add the following columns:
```Python
ohlc['timestamp'] = pd.to_datetime(ohlc['ticks'], unit='ms')
ohlc['instrument_name'] = "BTC-PERPETUAL"
ohlc['resolution'] = 60
```

Save `ohlc` into a table of the same name inside a database called "07_datam.db".


In [10]:
print(df.columns)
print(df['datetime'].head())

Index(['datetime', 'open', 'high', 'low', 'close', 'volume'], dtype='object')
0   2025-12-02 03:00:00+00:00
1   2025-12-02 04:00:00+00:00
2   2025-12-02 05:00:00+00:00
3   2025-12-02 06:00:00+00:00
4   2025-12-02 07:00:00+00:00
Name: datetime, dtype: datetime64[ns, UTC]


In [14]:
import pandas as pd

ohlc = pd.DataFrame({
    "timestamp":         pd.to_datetime(result["ticks"], unit="ms", utc=True),
    "open":             result["open"],
    "high":             result["high"],
    "low":              result["low"],
    "close":            result["close"],
    "volume":           result["volume"],
    "instrument_name":  "BTC-PERPETUAL",
    "resolution":       60
})

print(ohlc)

                     timestamp     open     high      low    close  \
0    2025-12-02 03:00:00+00:00  86554.5  87174.5  86221.0  86999.0   
1    2025-12-02 04:00:00+00:00  86999.0  87280.0  86920.5  86987.0   
2    2025-12-02 05:00:00+00:00  86986.5  87138.0  86849.0  87012.5   
3    2025-12-02 06:00:00+00:00  87027.5  87201.5  86874.0  87121.5   
4    2025-12-02 07:00:00+00:00  87116.5  87134.0  86901.5  87020.5   
...                        ...      ...      ...      ...      ...   
4996 2026-06-28 07:00:00+00:00  59943.0  60199.5  59939.0  60149.5   
4997 2026-06-28 08:00:00+00:00  60149.0  60472.0  60080.0  60305.0   
4998 2026-06-28 09:00:00+00:00  60302.0  60412.5  60195.0  60195.0   
4999 2026-06-28 10:00:00+00:00  60194.0  60230.0  60010.0  60092.0   
5000 2026-06-28 11:00:00+00:00  60092.5  60295.0  60048.0  60227.5   

          volume instrument_name  resolution  
0     209.233762   BTC-PERPETUAL          60  
1     206.927476   BTC-PERPETUAL          60  
2     117.700418  

In [15]:
# saving it into a table inside a database
import sqlite3

conn = sqlite3.connect("07_datam.db")  # creates/opens the .db file
ohlc.to_sql("ohlc", conn, if_exists="replace", index=False)  # saves df as table "ohlc"
conn.close()

In [16]:
conn = sqlite3.connect("07_datam.db")
result = pd.read_sql("SELECT * FROM ohlc", conn)
conn.close()
print(result)

                      timestamp     open     high      low    close  \
0     2025-12-02 03:00:00+00:00  86554.5  87174.5  86221.0  86999.0   
1     2025-12-02 04:00:00+00:00  86999.0  87280.0  86920.5  86987.0   
2     2025-12-02 05:00:00+00:00  86986.5  87138.0  86849.0  87012.5   
3     2025-12-02 06:00:00+00:00  87027.5  87201.5  86874.0  87121.5   
4     2025-12-02 07:00:00+00:00  87116.5  87134.0  86901.5  87020.5   
...                         ...      ...      ...      ...      ...   
4996  2026-06-28 07:00:00+00:00  59943.0  60199.5  59939.0  60149.5   
4997  2026-06-28 08:00:00+00:00  60149.0  60472.0  60080.0  60305.0   
4998  2026-06-28 09:00:00+00:00  60302.0  60412.5  60195.0  60195.0   
4999  2026-06-28 10:00:00+00:00  60194.0  60230.0  60010.0  60092.0   
5000  2026-06-28 11:00:00+00:00  60092.5  60295.0  60048.0  60227.5   

          volume instrument_name  resolution  
0     209.233762   BTC-PERPETUAL          60  
1     206.927476   BTC-PERPETUAL          60  
2     

### Task: 

Create a Python `class DataHandler` that connects to 07_datam.db. The class should have functions that:
* download and save,
* select and return, and
* display data as a plot (e.g. the last price over time).